# 02 — Train a batch of (arm, fold) runs

**One list entry = one run.** This notebook predates the per-arm-family split; the
sweep as actually run is `02a`–`02e` for the registered arms and `02f`–`02g` for the two
post-hoc ones, 48 runs in total. Fill `RUNS` in the parameters cell, then
*Save & Run All*. Each run writes its own `/kaggle/working/runs/<tag>/` directory, so one
session can carry many runs and a session that dies part-way keeps everything already
finished.

Nothing is tunable here. Every hyperparameter comes from `configs/model.yaml` in the pinned
commit and every decode setting from the frozen `DecodeConfig`. The arm name selects a data
file and nothing else — that is what makes every arm comparable.

**Attach:** dataset `emocap-v2-arms`; **Accelerator:** GPU **T4 x2**; **Internet:** on.

> P100 will not work. It is compute capability sm_60 and Kaggle's PyTorch build ships
> kernels for sm_70 and up, so every launch fails with `no kernel image is available`.

The registered 36 runs are: 6 arms × 5 folds, plus one negative control per arm on fold 0.

Each run writes `predictions.jsonl`, `predictions_novis.jsonl` (the same cells decoded
with the image blanked -- the registered visual-dependence probe), `stats.json`,
`manifest.json`, and `adapter.pt` (the trainable weights, ~35 MB).

Pull just the results, leaving the weights on Kaggle:

    kaggle kernels output <owner>/<slug> -p tmp/batch --page-size 200 \
        --file-pattern 'runs/.*\.(jsonl|json)$'

Or everything, weights included:

    kaggle kernels output <owner>/<slug> -p tmp/batch --page-size 200 --file-pattern 'runs/'

In [ ]:
# ── parameters ────────────────────────────────────────────────────────────
# (ARM, FOLD, NEGATIVE_CONTROL). Runs execute in order.
#   arms: S_paired25 | S_paired5 | S_unpaired | V1_paired5 | V1_unpaired | H_unpaired
#   folds: 0..4      negative control: fold 0 only, one per arm
#
# Group by cost. The three unpaired arms are ~3 min a run, so a dozen fit comfortably;
# S_paired25 is ~1.9 h a fold, so batch at most 5 of those against Kaggle's 12 h session cap.
RUNS = [
    ("S_unpaired",  1, False),
    ("S_unpaired",  2, False),
    ("S_unpaired",  3, False),
    ("S_unpaired",  4, False),
    ("V1_unpaired", 1, False),
    ("V1_unpaired", 2, False),
    ("V1_unpaired", 3, False),
    ("V1_unpaired", 4, False),
    ("H_unpaired",  1, False),
    ("H_unpaired",  2, False),
    ("H_unpaired",  3, False),
    ("H_unpaired",  4, False),
]
SEED = 42

DATA = "/kaggle/input/emocap-v2-arms"
OUT = "/kaggle/working/runs"

In [ ]:
# Clone the EXACT commit the data was built from. Pinning to the commit recorded in
# provenance.json is what stops a notebook from pairing this dataset version with a
# different version of the code -- a mismatch that would be silent and unrecoverable.
import json, os, subprocess
from pathlib import Path

COMMIT = json.loads(Path(DATA, "provenance.json").read_text())["git_commit"]
if not Path("/kaggle/working/EmoCap").exists():
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/asjad2401/EmoCap.git",
                    "/kaggle/working/EmoCap"], check=True)
subprocess.run(["git", "-C", "/kaggle/working/EmoCap", "checkout", "-q", COMMIT], check=True)
print("code at", COMMIT[:12])

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/EmoCap/src")
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
import time

results = []
t_all = time.time()

for i, (arm, fold, nc) in enumerate(RUNS, 1):
    tag = f"{arm}-f{fold}" + ("-nc" if nc else "")
    run_dir = Path(OUT, tag)

    # Resumable: a re-run of this notebook skips what already landed, so recovering from
    # a dead session costs only the runs that had not finished.
    if (run_dir / "predictions.jsonl").exists():
        print(f"[{i}/{len(RUNS)}] {tag}: already present, skipping\n", flush=True)
        results.append((tag, "skipped", 0.0))
        continue

    cmd = [sys.executable, "-u", "/kaggle/working/EmoCap/scripts/train_arm.py",
           "--arm", arm, "--fold", str(fold), "--seed", str(SEED),
           "--data-root", DATA, "--out-root", OUT]
    if nc:
        cmd.append("--negative-control")

    print(f"[{i}/{len(RUNS)}] {tag}   ({(time.time()-t_all)/60:.1f} min into batch)",
          flush=True)
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print("   ", line, end="")
    rc = proc.wait()

    # A failure does NOT abort the batch. The runs are independent, and losing eleven good
    # ones because the fourth crashed is exactly the failure this loop exists to avoid.
    # Every non-zero exit is collected and re-raised by the summary cell.
    status = "ok" if rc == 0 else f"FAILED rc={rc}"
    results.append((tag, status, round((time.time() - t0) / 60, 1)))
    print(f"    -> {status}   {results[-1][2]} min\n", flush=True)

print(f"batch finished in {(time.time()-t_all)/60:.1f} min")

In [ ]:
# What landed. `empty_captions` above zero means decode collapsed and that run is suspect.
print(f"{'run':<26} {'status':<14} {'min':>6}")
for tag, status, mins in results:
    print(f"{tag:<26} {status:<14} {mins:>6.1f}")

# Only successful runs are summarised -- a crashed run has no stats.json, and reading it
# would raise a FileNotFoundError that names the wrong problem.
for tag, status, _ in results:
    if status != "ok":
        continue
    st = json.loads(Path(OUT, tag, "stats.json").read_text())
    n = sum(1 for _ in Path(OUT, tag, "predictions.jsonl").open())
    print(f"\n{tag}: {n:,} decoded, {st['empty_captions']} empty, "
          f"loss {st['losses'][0]:.3f} -> {st['losses'][-1]:.3f}, "
          f"{st['wall_minutes']} min")

failed = [t for t, s, _ in results if s.startswith("FAILED")]
if failed:
    raise RuntimeError(f"{len(failed)} run(s) failed: {failed}")